# Distributions & Sobolev Spaces

## From Generalized Functions to the Rigorous Foundations of PDE Theory

Classical calculus demands smoothness: a function must be differentiable to have a derivative.
But **partial differential equations** routinely produce solutions with kinks, jumps, and singularities.
The theory of **distributions** (Laurent Schwartz, 1940s) extends differentiation to *all* locally integrable functions
and beyond, while **Sobolev spaces** provide the natural Hilbert/Banach space framework in which
weak (distributional) solutions of PDEs live.

This notebook builds the key ideas **from scratch**:

| Section | Topic |
|---------|-------|
| 1 | Motivation from PDEs |
| 2 | Test Functions & Distributions |
| 3 | Operations on Distributions |
| 4 | Weak Derivatives |
| 5 | Sobolev Spaces |
| 6 | Sobolev Embedding Theorems |
| 7 | Lax-Milgram & Weak Formulation of PDEs |

In [ ]:
%matplotlib inline
import numpy as np
from scipy import integrate, linalg, sparse
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

In [ ]:
# ── Global constants & styling ────────────────────────────────────────────────

FIGSIZE      = (10, 5)
FIGSIZE_WIDE = (12, 5)
FIGSIZE_SQ   = (6, 6)
FIGSIZE_TALL = (10, 7)

COLORS = {
    'blue':    '#2176AE',
    'orange':  '#F57C20',
    'green':   '#27AE60',
    'red':     '#E74C3C',
    'purple':  '#8E44AD',
    'teal':    '#17A589',
    'grey':    '#7F8C8D',
    'black':   '#2C3E50',
}
PALETTE = list(COLORS.values())

plt.rcParams.update({
    'figure.figsize': FIGSIZE,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'lines.linewidth': 2,
    'legend.fontsize': 10,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print('Setup complete.')

---
## 1. Motivation from PDEs

### 1.1 Why classical derivatives fall short

Consider the **1-D wave equation** governing a vibrating string:

$$\frac{\partial^2 u}{\partial t^2} = c^2 \frac{\partial^2 u}{\partial x^2}$$

d'Alembert's general solution is $u(x,t) = f(x - ct) + g(x + ct)$ for *arbitrary* functions $f, g$.
If $f$ has a kink (e.g., $f(s) = |s|$), then $f''$ does not exist classically at $s=0$,
yet physically the string *does* vibrate --- the kink simply propagates.

Similarly, in electrostatics the potential of a **point charge** satisfies

$$-\nabla^2 \phi = \rho, \qquad \rho(\mathbf{x}) = q\,\delta(\mathbf{x}),$$

where $\delta$ is the Dirac delta --- not a classical function at all.

### 1.2 The idea of weak solutions

Instead of requiring $u$ to be twice differentiable *pointwise*, we multiply by a smooth
"test function" $\varphi$ with compact support and integrate by parts, transferring derivatives
onto $\varphi$. A function $u$ is a **weak solution** of $-u'' = f$ if

$$\int u'(x)\,\varphi'(x)\,dx = \int f(x)\,\varphi(x)\,dx \qquad \forall\,\varphi \in C_c^\infty.$$

This formulation makes sense even when $u$ is only once (weakly) differentiable.

In [ ]:
# ── Motivating example: vibrating string with a kink ──────────────────────────
# f(s) = |s|  propagating to the right at speed c

c = 1.0
x = np.linspace(-3, 3, 1000)

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_WIDE)

for idx, t in enumerate([0.0, 0.8, 1.6]):
    ax = axes[idx]
    u = np.abs(x - c * t)              # travelling kink
    ax.plot(x, u, color=PALETTE[0], lw=2.5)
    ax.axvline(c * t, ls='--', color=PALETTE[3], lw=1, label='kink location')
    ax.set_title(f't = {t:.1f}')
    ax.set_xlabel('x')
    ax.set_ylabel('u(x, t)')
    ax.set_ylim(-0.2, 3.5)
    ax.legend(fontsize=9)

fig.suptitle('d\'Alembert solution $u = |x - ct|$: kink propagates but $u_{xx}$ undefined classically',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Test Functions & Distributions

### 2.1 The test function space $\mathcal{D}(\Omega)$

Let $\Omega \subseteq \mathbb{R}^n$ be open. Define

$$\mathcal{D}(\Omega) = C_c^\infty(\Omega) = \{\varphi \in C^\infty(\Omega) : \operatorname{supp}(\varphi) \text{ is compact and } \operatorname{supp}(\varphi) \subset \Omega\}.$$

The canonical example is the **standard mollifier** (bump function):

$$\psi(x) = \begin{cases} C\,\exp\!\left(-\dfrac{1}{1 - |x|^2}\right) & |x| < 1, \\ 0 & |x| \geq 1, \end{cases}$$

where $C$ is chosen so that $\int \psi = 1$. It is $C^\infty$ and compactly supported.

### 2.2 Distributions

A **distribution** $T \in \mathcal{D}'(\Omega)$ is a continuous linear functional on $\mathcal{D}(\Omega)$:

$$T : \mathcal{D}(\Omega) \to \mathbb{R}, \qquad \varphi \mapsto \langle T, \varphi \rangle.$$

Continuity means: if $\varphi_n \to \varphi$ in $\mathcal{D}(\Omega)$ (uniformly in all derivatives,
supports contained in a fixed compact set), then $\langle T, \varphi_n \rangle \to \langle T, \varphi \rangle$.

### 2.3 Important examples

| Distribution | Action on test function $\varphi$ |
|---|---|
| **Regular** ($f \in L^1_{\text{loc}}$) | $\langle T_f, \varphi \rangle = \int f(x)\varphi(x)\,dx$ |
| **Dirac delta** $\delta_a$ | $\langle \delta_a, \varphi \rangle = \varphi(a)$ |
| **Heaviside** $H(x)$ | $\langle T_H, \varphi \rangle = \int_0^\infty \varphi(x)\,dx$ |
| **Principal value** $\text{p.v.}\frac{1}{x}$ | $\langle \text{p.v.}\frac{1}{x}, \varphi \rangle = \lim_{\varepsilon \to 0^+} \int_{|x|>\varepsilon} \frac{\varphi(x)}{x}\,dx$ |

### 2.4 Tempered distributions $\mathcal{S}'(\mathbb{R}^n)$

The **Schwartz space** $\mathcal{S}(\mathbb{R}^n)$ consists of smooth functions that decay
faster than any polynomial (together with all derivatives). Its dual $\mathcal{S}'$ is the
space of **tempered distributions**, which is the natural domain for the Fourier transform.

In [ ]:
# ── Standard mollifier (bump function) ────────────────────────────────────────

def bump(x):
    """Standard mollifier with support on (-1, 1)."""
    out = np.zeros_like(x, dtype=float)
    mask = np.abs(x) < 1.0
    out[mask] = np.exp(-1.0 / (1.0 - x[mask]**2))
    # Normalize
    return out

def bump_normalized(x):
    """Normalized mollifier so that integral = 1."""
    raw = bump(x)
    dx = x[1] - x[0]
    norm = np.sum(raw) * dx
    if norm > 0:
        return raw / norm
    return raw

x = np.linspace(-1.5, 1.5, 2000)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, bump_normalized(x), color=PALETTE[0], lw=2.5, label=r'$\psi(x) \propto e^{-1/(1-x^2)}$')
ax.fill_between(x, bump_normalized(x), alpha=0.15, color=PALETTE[0])
ax.set_title(r'Standard mollifier $\psi \in C_c^\infty(-1,1)$')
ax.set_xlabel('x')
ax.set_ylabel(r'$\psi(x)$')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Approximating the Dirac delta with Gaussian nascent deltas ────────────────
# phi_eps(x) = (1 / (eps * sqrt(2*pi))) * exp(-x^2 / (2*eps^2))
# As eps -> 0, phi_eps -> delta  in the sense of distributions.

def gaussian_nascent_delta(x, eps):
    """Gaussian approximation to delta: phi_eps(x)."""
    return np.exp(-x**2 / (2 * eps**2)) / (eps * np.sqrt(2 * np.pi))

def sinc_nascent_delta(x, eps):
    """Sinc-kernel approximation to delta."""
    return np.sinc(x / eps) / eps

x = np.linspace(-3, 3, 2000)
epsilons = [1.0, 0.5, 0.2, 0.05]

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)

# Panel 1: Gaussian nascent deltas
ax = axes[0]
for i, eps in enumerate(epsilons):
    ax.plot(x, gaussian_nascent_delta(x, eps), color=PALETTE[i],
            label=rf'$\varepsilon = {eps}$')
ax.set_title(r'Gaussian nascent deltas $\varphi_\varepsilon(x)$')
ax.set_xlabel('x')
ax.set_ylabel(r'$\varphi_\varepsilon(x)$')
ax.set_ylim(-0.5, 10)
ax.legend()

# Panel 2: Sinc nascent deltas
ax = axes[1]
for i, eps in enumerate(epsilons):
    ax.plot(x, sinc_nascent_delta(x, eps), color=PALETTE[i],
            label=rf'$\varepsilon = {eps}$')
ax.set_title(r'Sinc nascent deltas $\frac{1}{\varepsilon}\mathrm{sinc}(x/\varepsilon)$')
ax.set_xlabel('x')
ax.set_ylabel(r'$\varphi_\varepsilon(x)$')
ax.set_ylim(-5, 22)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Verify sifting property: integral phi_eps(x) f(x) dx  -->  f(0) ──────────

def test_function_f(x):
    """A smooth test function: f(x) = cos(x) * exp(-x^2/4)."""
    return np.cos(x) * np.exp(-x**2 / 4)

x_fine = np.linspace(-10, 10, 50000)
dx = x_fine[1] - x_fine[0]
f_vals = test_function_f(x_fine)
f_at_zero = test_function_f(0.0)  # = 1.0

eps_range = np.logspace(-0.2, -2.5, 40)
integrals = []
for eps in eps_range:
    phi_eps = gaussian_nascent_delta(x_fine, eps)
    integrals.append(np.sum(phi_eps * f_vals) * dx)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(eps_range, integrals, 'o-', color=PALETTE[0], markersize=4,
            label=r'$\int \varphi_\varepsilon(x)\,f(x)\,dx$')
ax.axhline(f_at_zero, ls='--', color=PALETTE[3], lw=1.5, label=f'$f(0) = {f_at_zero:.4f}$')
ax.set_xlabel(r'$\varepsilon$')
ax.set_ylabel('Integral value')
ax.set_title(r'Sifting property: $\langle \varphi_\varepsilon, f \rangle \to f(0)$ as $\varepsilon \to 0$')
ax.legend()
ax.invert_xaxis()
plt.tight_layout()
plt.show()

print(f'f(0) = {f_at_zero:.6f}')
print(f'Integral at eps = {eps_range[-1]:.4f}: {integrals[-1]:.6f}')
print(f'Error: {abs(integrals[-1] - f_at_zero):.2e}')

---
## 3. Operations on Distributions

### 3.1 Differentiation of distributions

The **distributional derivative** of $T \in \mathcal{D}'$ is defined by transferring the
derivative to the test function via integration by parts (no boundary terms, since $\varphi$
has compact support):

$$\langle T', \varphi \rangle = -\langle T, \varphi' \rangle \qquad \forall\,\varphi \in \mathcal{D}.$$

More generally, for multi-index $\alpha$:

$$\langle D^\alpha T, \varphi \rangle = (-1)^{|\alpha|} \langle T, D^\alpha \varphi \rangle.$$

**Key consequence**: every distribution is infinitely differentiable in the distributional sense!

### 3.2 Classical examples

- $|x|' = \operatorname{sgn}(x)$ (sign function)
- $H'(x) = \delta(x)$ (Heaviside $\to$ Dirac delta)
- $\delta'(x)$ acts by $\langle \delta', \varphi \rangle = -\varphi'(0)$

**Proof that $|x|' = \operatorname{sgn}(x)$:** For any test function $\varphi$,

$$\langle |x|', \varphi \rangle = -\langle |x|, \varphi' \rangle = -\int_{-\infty}^{\infty} |x|\,\varphi'(x)\,dx = -\int_{-\infty}^{0} (-x)\varphi'(x)\,dx - \int_0^\infty x\,\varphi'(x)\,dx.$$

Integrating each piece by parts (boundary terms vanish):

$$= -\int_{-\infty}^0 (-1)\varphi(x)\,dx - \int_0^\infty \varphi(x)\,dx = \int_{-\infty}^0 (-1)\varphi(x)\,dx + \int_0^\infty (1)\varphi(x)\,dx $$

Wait -- let's be more careful.  Integrate by parts on $(-\infty, 0)$:

$$-\int_{-\infty}^0 (-x)\varphi'(x)\,dx = -\Big[(-x)\varphi(x)\Big]_{-\infty}^0 + \int_{-\infty}^0 (-1)\varphi(x)\,dx = -\int_{-\infty}^0 \varphi(x)\,dx.$$

On $(0, \infty)$:

$$-\int_0^\infty x\,\varphi'(x)\,dx = -\Big[x\,\varphi(x)\Big]_0^\infty + \int_0^\infty \varphi(x)\,dx = \int_0^\infty \varphi(x)\,dx.$$

Combining:

$$\langle |x|', \varphi \rangle = -\int_{-\infty}^0 \varphi(x)\,dx + \int_0^\infty \varphi(x)\,dx = \int_{-\infty}^\infty \operatorname{sgn}(x)\,\varphi(x)\,dx = \langle \operatorname{sgn}, \varphi \rangle. \;\; \square$$

### 3.3 Convolution

If $T$ is a distribution and $\psi \in \mathcal{D}$, the convolution $T * \psi$ is a smooth function:

$$(T * \psi)(x) = \langle T_y, \psi(x - y) \rangle.$$

In particular, convolution with a **mollifier** $\varphi_\varepsilon(x) = \varepsilon^{-1}\varphi(x/\varepsilon)$
gives $f * \varphi_\varepsilon \to f$ in $L^p$ as $\varepsilon \to 0$.

### 3.4 Fourier transform of distributions

For tempered distributions $T \in \mathcal{S}'$:

$$\langle \widehat{T}, \varphi \rangle = \langle T, \widehat{\varphi} \rangle \qquad \forall\,\varphi \in \mathcal{S}.$$

Important results:
- $\widehat{\delta} = 1$ (Fourier transform of delta is the constant function 1)
- $\widehat{1} = 2\pi\,\delta$ (Fourier transform of constant is a delta)

In [ ]:
# ── Distributional derivative of |x| via finite differences ──────────────────
# As h -> 0, the finite difference (|x+h| - |x-h|) / (2h) --> sgn(x)

x = np.linspace(-2, 2, 2000)
abs_x = np.abs(x)
sgn_x = np.sign(x)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)

# Panel 1: finite difference approximations to |x|'
ax = axes[0]
h_vals = [0.5, 0.2, 0.05, 0.01]
for i, h in enumerate(h_vals):
    fd = (np.abs(x + h) - np.abs(x - h)) / (2 * h)
    ax.plot(x, fd, color=PALETTE[i], alpha=0.8, label=f'h = {h}')
ax.plot(x, sgn_x, 'k--', lw=1.5, label=r'$\mathrm{sgn}(x)$')
ax.set_title(r"Finite differences of $|x|$ $\to$ $\mathrm{sgn}(x)$")
ax.set_xlabel('x')
ax.legend()

# Panel 2: finite difference approximations to H(x)' -> delta(x)
ax = axes[1]
heaviside = np.where(x >= 0, 1.0, 0.0)
for i, h in enumerate(h_vals):
    H_plus  = np.where(x + h >= 0, 1.0, 0.0)
    H_minus = np.where(x - h >= 0, 1.0, 0.0)
    fd = (H_plus - H_minus) / (2 * h)
    ax.plot(x, fd, color=PALETTE[i], alpha=0.8, label=f'h = {h}')
ax.set_title(r"Finite differences of $H(x)$ $\to$ $\delta(x)$")
ax.set_xlabel('x')
ax.set_ylim(-1, 55)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Convolution with mollifier as smoothing operation ─────────────────────────
# Convolve a rough function with Gaussian mollifiers of decreasing width

def rough_function(x):
    """A piecewise function with jumps and kinks."""
    y = np.zeros_like(x)
    y += np.where(x < -1, -1.0, 0.0)
    y += np.where((x >= -1) & (x < 0), x + 1, 0.0)     # ramp from 0 to 1
    y += np.where((x >= 0) & (x < 1), 1.0, 0.0)         # constant 1
    y += np.where(x >= 1, -0.5, 0.0)                     # jump down to -0.5
    return y

def convolve_with_gaussian(f_vals, x, sigma):
    """Convolve discrete f with a Gaussian kernel of width sigma."""
    dx = x[1] - x[0]
    # Build Gaussian kernel centered at 0
    kernel_half_width = int(5 * sigma / dx)
    k_x = np.arange(-kernel_half_width, kernel_half_width + 1) * dx
    kernel = np.exp(-k_x**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi))
    kernel *= dx  # discretize the integral
    # Convolve (mode='same' keeps the array length)
    return np.convolve(f_vals, kernel, mode='same')

x = np.linspace(-3, 3, 4000)
f_rough = rough_function(x)

sigmas = [0.02, 0.1, 0.3, 0.8]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, f_rough, 'k-', lw=2, label='Original $f$')
for i, sigma in enumerate(sigmas):
    f_smooth = convolve_with_gaussian(f_rough, x, sigma)
    ax.plot(x, f_smooth, color=PALETTE[i], alpha=0.8,
            label=rf'$f * \varphi_\sigma$, $\sigma={sigma}$')

ax.set_title('Mollification: convolution smooths rough functions')
ax.set_xlabel('x')
ax.set_ylabel('$f(x)$')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Weak Derivatives

### 4.1 Definition

Let $f \in L^1_{\text{loc}}(\Omega)$. A function $g \in L^1_{\text{loc}}(\Omega)$ is the **weak derivative**
of $f$ (written $f' = g$ weakly) if

$$\int_\Omega f(x)\,\varphi'(x)\,dx = -\int_\Omega g(x)\,\varphi(x)\,dx \qquad \forall\,\varphi \in C_c^\infty(\Omega).$$

This is precisely the distributional derivative, restricted to the case where $T_f'$ happens
to be a *regular* distribution (i.e., representable by an $L^1_{\text{loc}}$ function).

### 4.2 Key properties

- **Uniqueness**: Weak derivatives are unique (up to sets of measure zero).
- **Classical $\Rightarrow$ weak**: If $f$ is classically differentiable, the weak derivative equals the classical one.
- **Weak $\not\Rightarrow$ classical**: The function $f(x) = |x|$ has weak derivative $g(x) = \operatorname{sgn}(x)$, but $f$ is not classically differentiable at $x = 0$.

### 4.3 Example: $f(x) = |x|$ on $[-1, 1]$

We verify numerically that $\int_{-1}^1 |x|\,\varphi'(x)\,dx = -\int_{-1}^1 \operatorname{sgn}(x)\,\varphi(x)\,dx$
for several test functions $\varphi \in C_c^\infty(-1, 1)$.

In [ ]:
# ── Numerical verification of weak derivative: |x|' = sgn(x) ─────────────────

def make_test_function(x, center=0.0, width=0.5):
    """A smooth compactly supported test function (rescaled bump)."""
    t = (x - center) / width
    phi = np.zeros_like(x)
    mask = np.abs(t) < 1.0
    phi[mask] = np.exp(-1.0 / (1.0 - t[mask]**2))
    return phi

def numerical_derivative(y, dx):
    """Central finite difference."""
    dy = np.zeros_like(y)
    dy[1:-1] = (y[2:] - y[:-2]) / (2 * dx)
    dy[0] = (y[1] - y[0]) / dx
    dy[-1] = (y[-1] - y[-2]) / dx
    return dy

x = np.linspace(-1, 1, 10000)
dx = x[1] - x[0]
f = np.abs(x)            # f(x) = |x|
g = np.sign(x)           # proposed weak derivative = sgn(x)

# Test with several test functions at different centers
print('Verification of weak derivative:  integral f*phi\' dx  vs  -integral g*phi dx')
print('=' * 72)
print(f'{"Center":>8}  {"Width":>8}  {"int f*phi\' dx":>16}  {"-int g*phi dx":>16}  {"Error":>12}')
print('-' * 72)

test_configs = [
    (0.0, 0.5),
    (0.3, 0.4),
    (-0.2, 0.3),
    (0.5, 0.3),
    (-0.5, 0.4),
]

for center, width in test_configs:
    phi = make_test_function(x, center, width)
    dphi = numerical_derivative(phi, dx)
    
    # Left side: integral of f * phi'
    lhs = np.trapz(f * dphi, x)
    # Right side: -integral of g * phi
    rhs = -np.trapz(g * phi, x)
    
    print(f'{center:8.2f}  {width:8.2f}  {lhs:16.8f}  {rhs:16.8f}  {abs(lhs - rhs):12.2e}')

In [ ]:
# ── Visualize the weak derivative relationship ───────────────────────────────

x = np.linspace(-1, 1, 2000)
dx = x[1] - x[0]

fig, axes = plt.subplots(1, 3, figsize=FIGSIZE_WIDE)

# f(x) = |x|
ax = axes[0]
ax.plot(x, np.abs(x), color=PALETTE[0], lw=2.5)
ax.set_title(r'$f(x) = |x|$')
ax.set_xlabel('x')

# Weak derivative sgn(x)
ax = axes[1]
ax.plot(x, np.sign(x), color=PALETTE[1], lw=2.5)
ax.set_title(r"Weak derivative: $f'(x) = \mathrm{sgn}(x)$")
ax.set_xlabel('x')
ax.set_ylim(-1.5, 1.5)

# Test function and integrands
ax = axes[2]
phi = make_test_function(x, center=0.0, width=0.6)
dphi = numerical_derivative(phi, dx)
ax.plot(x, np.abs(x) * dphi, color=PALETTE[0], label=r'$|x|\cdot\varphi\'(x)$')
ax.plot(x, -np.sign(x) * phi, color=PALETTE[3], ls='--',
        label=r'$-\mathrm{sgn}(x)\cdot\varphi(x)$')
ax.fill_between(x, np.abs(x) * dphi, alpha=0.1, color=PALETTE[0])
ax.fill_between(x, -np.sign(x) * phi, alpha=0.1, color=PALETTE[3])
ax.set_title('Integrands (areas must match)')
ax.set_xlabel('x')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 5. Sobolev Spaces

### 5.1 Definition

For an open domain $\Omega \subseteq \mathbb{R}^n$, integer $k \geq 0$, and $1 \leq p \leq \infty$,
the **Sobolev space** is

$$W^{k,p}(\Omega) = \{u \in L^p(\Omega) : D^\alpha u \in L^p(\Omega) \text{ for all } |\alpha| \leq k\},$$

where $D^\alpha u$ denotes the **weak** (distributional) derivative.

### 5.2 Sobolev norm

$$\|u\|_{W^{k,p}} = \left(\sum_{|\alpha| \leq k} \|D^\alpha u\|_{L^p}^p\right)^{1/p} \quad (1 \leq p < \infty).$$

### 5.3 The Hilbert case: $H^k(\Omega) = W^{k,2}(\Omega)$

When $p = 2$, we write $H^k(\Omega) = W^{k,2}(\Omega)$. This is a **Hilbert space** with inner product

$$(u, v)_{H^k} = \sum_{|\alpha| \leq k} \int_\Omega D^\alpha u \cdot D^\alpha v \, dx,$$

and the associated norm:

$$\|u\|_{H^k} = \left(\sum_{|\alpha| \leq k} \|D^\alpha u\|_{L^2}^2\right)^{1/2}.$$

### 5.4 Chain of inclusions

$$\cdots \subset H^2(\Omega) \subset H^1(\Omega) \subset L^2(\Omega) = H^0(\Omega).$$

Higher Sobolev regularity means more (weak) derivatives are square-integrable.

### 5.5 Completeness

$W^{k,p}(\Omega)$ is a **Banach space** (complete normed vector space). For $p=2$, $H^k(\Omega)$
is a **Hilbert space**. This completeness is crucial: it guarantees that limits of Cauchy
sequences of approximate solutions remain in the space.

In [ ]:
# ── Computing Sobolev norms numerically ──────────────────────────────────────

def L2_norm(f_vals, dx):
    """Compute L^2 norm from discrete samples."""
    return np.sqrt(np.trapz(f_vals**2, dx=dx))

def H1_norm(f_vals, df_vals, dx):
    """Compute H^1 norm: sqrt(||f||^2_{L^2} + ||f'||^2_{L^2})."""
    return np.sqrt(np.trapz(f_vals**2, dx=dx) + np.trapz(df_vals**2, dx=dx))

def H2_norm(f_vals, df_vals, d2f_vals, dx):
    """Compute H^2 norm: sqrt(||f||^2 + ||f'||^2 + ||f''||^2)."""
    return np.sqrt(
        np.trapz(f_vals**2, dx=dx)
        + np.trapz(df_vals**2, dx=dx)
        + np.trapz(d2f_vals**2, dx=dx)
    )

x = np.linspace(-1, 1, 10000)
dx = x[1] - x[0]

# ── Sample functions ──

# 1. Smooth function: sin(pi*x)
f1 = np.sin(np.pi * x)
df1 = np.pi * np.cos(np.pi * x)
d2f1 = -np.pi**2 * np.sin(np.pi * x)

# 2. Hat function: 1 - |x|  (in H^1 but NOT in H^2, since derivative is sgn with a jump)
f2 = np.maximum(1.0 - np.abs(x), 0.0)
df2 = np.where(x < 0, 1.0, np.where(x > 0, -1.0, 0.0))  # weak derivative
df2[np.abs(x) > 1] = 0.0
# d2f2 = -2*delta(x)  -- not in L^2, so H^2 norm is infinite
d2f2_approx = numerical_derivative(df2, dx)  # will have a huge spike at x=0

# 3. Smooth bump
f3 = np.exp(-5 * x**2)
df3 = -10 * x * np.exp(-5 * x**2)
d2f3 = (-10 + 100 * x**2) * np.exp(-5 * x**2)

print('Sobolev norms on [-1, 1]')
print('=' * 65)
print(f'{"Function":>20}  {"||f||_L2":>10}  {"||f||_H1":>10}  {"||f||_H2":>10}')
print('-' * 65)

for name, fv, dfv, d2fv in [
    ('sin(pi*x)', f1, df1, d2f1),
    ('hat: 1-|x|', f2, df2, d2f2_approx),
    ('exp(-5x^2)', f3, df3, d2f3),
]:
    l2 = L2_norm(fv, dx)
    h1 = H1_norm(fv, dfv, dx)
    h2 = H2_norm(fv, dfv, d2fv, dx)
    print(f'{name:>20}  {l2:10.4f}  {h1:10.4f}  {h2:10.4f}')

print()
print('Note: The hat function\'s H^2 "norm" is mesh-dependent (diverges as dx -> 0)')
print('because its second weak derivative is -2*delta(x), which is not in L^2.')
print('This shows: hat function is in H^1 but NOT in H^2.')

In [ ]:
# ── Demonstrate mesh-dependence of H^2 "norm" for the hat function ───────────
# As mesh refines, the H^2 norm of hat(x) diverges, confirming it is not in H^2

N_vals = [100, 500, 1000, 5000, 10000, 50000]
h2_hat_norms = []
h1_hat_norms = []

for N in N_vals:
    xn = np.linspace(-1, 1, N)
    dxn = xn[1] - xn[0]
    fn = np.maximum(1.0 - np.abs(xn), 0.0)
    dfn = np.where(xn < 0, 1.0, np.where(xn > 0, -1.0, 0.0))
    dfn[np.abs(xn) > 1] = 0.0
    d2fn = numerical_derivative(dfn, dxn)
    
    h1_hat_norms.append(H1_norm(fn, dfn, dxn))
    h2_hat_norms.append(H2_norm(fn, dfn, d2fn, dxn))

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)

ax = axes[0]
ax.semilogx(N_vals, h1_hat_norms, 'o-', color=PALETTE[2], lw=2, markersize=7)
ax.set_xlabel('Number of grid points $N$')
ax.set_ylabel(r'$\|\mathrm{hat}\|_{H^1}$')
ax.set_title(r'$H^1$ norm of $1 - |x|$ (converges $\Rightarrow$ in $H^1$)')

ax = axes[1]
ax.loglog(N_vals, h2_hat_norms, 's-', color=PALETTE[3], lw=2, markersize=7)
ax.set_xlabel('Number of grid points $N$')
ax.set_ylabel(r'$\|\mathrm{hat}\|_{H^2}$')
ax.set_title(r'$H^2$ "norm" of $1 - |x|$ (diverges $\Rightarrow$ NOT in $H^2$)')

plt.tight_layout()
plt.show()

---
## 6. Sobolev Embedding Theorems

### 6.1 The main embedding result

**Sobolev Embedding Theorem.** Let $\Omega \subseteq \mathbb{R}^n$ be a bounded domain with
Lipschitz boundary. If

$$k - \frac{n}{p} > m,$$

then $W^{k,p}(\Omega) \hookrightarrow C^m(\bar{\Omega})$ (continuous embedding).

**Interpretation:** Having enough weak derivatives in $L^p$ forces the function to be
classically smooth. More Sobolev regularity $\Rightarrow$ more classical smoothness.

**Example (1D, $p = 2$, $n = 1$):** $H^k(\Omega) \hookrightarrow C^m$ when $k - 1/2 > m$, i.e., $k \geq m + 1$.
So $H^1 \hookrightarrow C^0$ (continuous), $H^2 \hookrightarrow C^1$ (continuously differentiable), etc.

### 6.2 Rellich-Kondrachov compactness

If $kp > n$, the embedding $W^{k,p}(\Omega) \hookrightarrow C(\bar{\Omega})$ is **compact**
(bounded sequences have convergent subsequences). This is vital for proving existence
of PDE solutions via compactness arguments.

### 6.3 Morrey's inequality (1D)

For $n = 1$, every $u \in H^1(a,b)$ is (after modification on a null set) absolutely
continuous, and

$$\|u\|_{L^\infty} \leq C\,\|u\|_{H^1}.$$

In [ ]:
# ── Visualize Sobolev regularity: functions at different H^k levels ───────────
# We use Fourier series: u(x) = sum a_n sin(n*pi*x) on [0,1]
# The H^k norm of u is  sum |a_n|^2 (1 + (n*pi)^2 + ... + (n*pi)^{2k})
# So coefficients a_n ~ n^{-alpha} give u in H^k iff alpha > k + 1/2.

np.random.seed(42)
x = np.linspace(0, 1, 2000)
N_modes = 200
n_arr = np.arange(1, N_modes + 1)

# Random phases
phases = np.random.uniform(0, 2 * np.pi, N_modes)

def build_fourier_function(x, alpha, n_arr, phases):
    """Build u(x) = sum (1/n^alpha) * sin(n*pi*x + phase_n)."""
    u = np.zeros_like(x)
    for i, n in enumerate(n_arr):
        u += (1.0 / n**alpha) * np.sin(n * np.pi * x + phases[i])
    return u

# alpha values corresponding to different Sobolev regularity:
# u in H^k  <=> alpha > k + 1/2
# alpha=1.0 => H^0 (L^2) only, barely  [since 1.0 > 0 + 0.5]
# alpha=1.5 => in H^0 but barely above threshold for H^1 (need >1.5)
# alpha=2.0 => in H^1 (2.0 > 1.5)
# alpha=3.0 => in H^2 (3.0 > 2.5)
# alpha=4.0 => in H^3 (4.0 > 3.5)

alphas = [1.0, 1.5, 2.5, 4.0]
labels = [
    r'$\alpha = 1.0$: in $H^0 = L^2$ (rough)',
    r'$\alpha = 1.5$: barely in $H^0$',
    r'$\alpha = 2.5$: in $H^1$ (continuous)',
    r'$\alpha = 4.0$: in $H^3$ (very smooth)',
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, (alpha, label) in enumerate(zip(alphas, labels)):
    ax = axes.flat[idx]
    u = build_fourier_function(x, alpha, n_arr, phases)
    ax.plot(x, u, color=PALETTE[idx], lw=1.5)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('x')
    ax.set_ylabel('u(x)')

fig.suptitle(
    r'Sobolev embedding: $a_n \sim n^{-\alpha}$ $\Rightarrow$ higher $\alpha$ = more regular',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Quantify: compute H^k norms for each function to verify embedding ────────
# For u(x) = sum a_n sin(n*pi*x), the k-th derivative involves (n*pi)^k.
# ||D^k u||^2_{L^2} = (1/2) sum a_n^2 (n*pi)^{2k}  on [0,1].

print('Sobolev norms (analytical, Fourier coefficients a_n = n^{-alpha})')
print('=' * 75)
print(f'{"alpha":>8}  {"||u||_L2":>12}  {"||u||_H1":>12}  {"||u||_H2":>12}  {"||u||_H3":>12}')
print('-' * 75)

for alpha in alphas:
    a_n = 1.0 / n_arr**alpha
    norms = []
    for k in range(4):  # H^0, H^1, H^2, H^3
        # ||u||^2_{H^k} = sum_{j=0}^{k} ||D^j u||^2_{L^2}
        #               = sum_{j=0}^{k} (1/2) sum_n a_n^2 (n*pi)^{2j}
        hk_sq = 0.0
        for j in range(k + 1):
            hk_sq += 0.5 * np.sum(a_n**2 * (n_arr * np.pi)**(2 * j))
        if np.isfinite(hk_sq) and hk_sq < 1e20:
            norms.append(f'{np.sqrt(hk_sq):12.4f}')
        else:
            norms.append(f'{"DIVERGES":>12}')
    print(f'{alpha:8.1f}  {norms[0]}  {norms[1]}  {norms[2]}  {norms[3]}')

print()
print('A finite norm at level H^k confirms membership; divergence means exclusion.')

---
## 7. Lax-Milgram Theorem & Weak Formulation of PDEs

### 7.1 Bilinear forms and the abstract framework

Let $V$ be a Hilbert space. A **bilinear form** $a: V \times V \to \mathbb{R}$ is:

- **Continuous (bounded)**: $|a(u, v)| \leq M\,\|u\|_V\,\|v\|_V$ for some $M > 0$.
- **Coercive**: $a(u, u) \geq \alpha\,\|u\|_V^2$ for some $\alpha > 0$.

### 7.2 The Lax-Milgram Theorem

**Theorem (Lax-Milgram).** If $a(\cdot, \cdot)$ is a continuous, coercive bilinear form on
a Hilbert space $V$, and $F \in V'$ is a continuous linear functional, then there exists
a **unique** $u \in V$ such that

$$a(u, v) = F(v) \qquad \forall\, v \in V.$$

Moreover, $\|u\|_V \leq \frac{1}{\alpha}\|F\|_{V'}$.

This is a powerful generalization of the Riesz representation theorem (which handles
the case $a(u,v) = (u,v)_V$).

### 7.3 Application: weak formulation of $-u'' = f$

Consider the **Poisson problem** on $[0, 1]$ with homogeneous Dirichlet boundary conditions:

$$-u''(x) = f(x), \quad x \in (0, 1), \qquad u(0) = u(1) = 0.$$

Multiply by a test function $v \in H^1_0(0,1)$ and integrate by parts:

$$\int_0^1 u'(x)\,v'(x)\,dx = \int_0^1 f(x)\,v(x)\,dx \qquad \forall\, v \in H^1_0(0,1).$$

Define:
- $V = H^1_0(0,1)$ (Sobolev functions vanishing at endpoints)
- $a(u, v) = \int_0^1 u'v'\,dx$ (bilinear form)
- $F(v) = \int_0^1 fv\,dx$ (linear functional)

**Coercivity** follows from the **Poincare inequality**: for $u \in H^1_0(0,1)$,

$$\|u\|_{L^2} \leq C_P \|u'\|_{L^2},$$

so $a(u,u) = \|u'\|^2_{L^2} \geq \frac{1}{1 + C_P^2}\|u\|^2_{H^1}$.

By Lax-Milgram, there is a **unique weak solution** $u \in H^1_0(0,1)$.

### 7.4 Finite Element Method (FEM)

We approximate $V = H^1_0$ by a finite-dimensional subspace $V_h$ spanned by
**piecewise linear hat functions**:

$$\phi_i(x) = \begin{cases} (x - x_{i-1})/h & x \in [x_{i-1}, x_i], \\ (x_{i+1} - x)/h & x \in [x_i, x_{i+1}], \\ 0 & \text{otherwise}, \end{cases}$$

where $x_0 = 0 < x_1 < \cdots < x_{N} < x_{N+1} = 1$ with mesh size $h = 1/(N+1)$.

Writing $u_h = \sum_{j=1}^{N} u_j \phi_j$, the weak formulation becomes the linear system

$$K \mathbf{u} = \mathbf{f},$$

where the **stiffness matrix** $K_{ij} = a(\phi_j, \phi_i) = \int_0^1 \phi_j'\phi_i'\,dx$
and **load vector** $f_i = \int_0^1 f\,\phi_i\,dx$.

In [ ]:
# ── 1D Finite Element Method from scratch ────────────────────────────────────
# Solve: -u''(x) = f(x) on [0,1], u(0) = u(1) = 0

def fem_1d(f_func, N, return_mesh=False):
    """
    Solve -u'' = f on [0,1] with u(0)=u(1)=0 using piecewise linear FEM.
    
    Parameters
    ----------
    f_func : callable
        Right-hand side function f(x).
    N : int
        Number of interior nodes (mesh has N+2 total points including endpoints).
    return_mesh : bool
        If True, also return the full mesh and solution arrays.
    
    Returns
    -------
    x_full : ndarray, shape (N+2,)
        Full mesh including boundary.
    u_full : ndarray, shape (N+2,)
        FEM solution including boundary values (0).
    """
    h = 1.0 / (N + 1)  # mesh size
    x_nodes = np.linspace(0, 1, N + 2)  # includes boundary
    x_int = x_nodes[1:-1]  # interior nodes
    
    # ── Assemble stiffness matrix K (tridiagonal) ──
    # K_ii = integral phi_i' * phi_i' dx = 2/h
    # K_{i,i+1} = K_{i+1,i} = integral phi_i' * phi_{i+1}' dx = -1/h
    
    main_diag = np.full(N, 2.0 / h)
    off_diag = np.full(N - 1, -1.0 / h)
    
    K = sparse.diags(
        [off_diag, main_diag, off_diag],
        offsets=[-1, 0, 1],
        shape=(N, N),
        format='csc'
    )
    
    # ── Assemble load vector f_i = integral f(x) phi_i(x) dx ──
    # Using midpoint quadrature on each element:
    # integral_{x_{i-1}}^{x_i} f * phi_i dx  approx  f(midpoint) * (h/2) * 1/2 ... 
    # More accurate: use Simpson's rule or exact for simple f.
    # For piecewise linear hat, exact integral with trapezoidal rule:
    # f_i = h * f(x_i)  (lumped)  -- good enough for smooth f.
    
    # Better: 2-point Gauss quadrature on each element
    rhs = np.zeros(N)
    for i in range(N):
        xi = x_int[i]
        # Left element [x_{i-1}, x_i] = [xi - h, xi]
        # phi_i(x) = (x - (xi - h)) / h on this element
        # Use 2-point Gauss quadrature on [xi-h, xi]
        gp1 = (xi - h) + h * (0.5 - np.sqrt(3)/6)  # Gauss point 1
        gp2 = (xi - h) + h * (0.5 + np.sqrt(3)/6)  # Gauss point 2
        phi_gp1 = (gp1 - (xi - h)) / h
        phi_gp2 = (gp2 - (xi - h)) / h
        rhs[i] += (h / 2) * (f_func(gp1) * phi_gp1 + f_func(gp2) * phi_gp2)
        
        # Right element [x_i, x_{i+1}] = [xi, xi + h]
        # phi_i(x) = ((xi + h) - x) / h on this element
        gp1 = xi + h * (0.5 - np.sqrt(3)/6)
        gp2 = xi + h * (0.5 + np.sqrt(3)/6)
        phi_gp1 = ((xi + h) - gp1) / h
        phi_gp2 = ((xi + h) - gp2) / h
        rhs[i] += (h / 2) * (f_func(gp1) * phi_gp1 + f_func(gp2) * phi_gp2)
    
    # ── Solve Ku = f ──
    u_int = spsolve(K, rhs)
    
    # Assemble full solution with boundary values
    u_full = np.zeros(N + 2)
    u_full[1:-1] = u_int
    
    return x_nodes, u_full

In [ ]:
# ── Example 1: f(x) = sin(pi*x), exact solution u(x) = sin(pi*x) / pi^2 ────

f_func = lambda x: np.sin(np.pi * x)
u_exact_func = lambda x: np.sin(np.pi * x) / np.pi**2

x_fine = np.linspace(0, 1, 1000)
u_exact = u_exact_func(x_fine)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)

# Panel 1: FEM solutions at different mesh sizes
ax = axes[0]
ax.plot(x_fine, u_exact, 'k-', lw=2.5, label='Exact')
for i, N in enumerate([4, 8, 20]):
    x_fem, u_fem = fem_1d(f_func, N)
    ax.plot(x_fem, u_fem, 'o--', color=PALETTE[i], markersize=5, label=f'FEM, N={N}')
ax.set_title(r'FEM solution of $-u\prime\prime = \sin(\pi x)$')
ax.set_xlabel('x')
ax.set_ylabel('u(x)')
ax.legend()

# Panel 2: Error vs mesh size (convergence)
ax = axes[1]
N_values = [4, 8, 16, 32, 64, 128, 256]
errors_Linf = []
errors_L2 = []
h_values = []

for N in N_values:
    x_fem, u_fem = fem_1d(f_func, N)
    u_ex_at_nodes = u_exact_func(x_fem)
    err = np.abs(u_fem - u_ex_at_nodes)
    errors_Linf.append(np.max(err))
    h = 1.0 / (N + 1)
    h_values.append(h)
    # L2 error
    errors_L2.append(np.sqrt(np.trapz(err**2, x_fem)))

h_arr = np.array(h_values)
ax.loglog(h_arr, errors_Linf, 'o-', color=PALETTE[0], lw=2, label=r'$\|e\|_{L^\infty}$')
ax.loglog(h_arr, errors_L2, 's-', color=PALETTE[1], lw=2, label=r'$\|e\|_{L^2}$')
# Reference slopes
ax.loglog(h_arr, 0.15 * h_arr**2, 'k--', lw=1, alpha=0.5, label=r'$O(h^2)$')
ax.set_xlabel('Mesh size $h$')
ax.set_ylabel('Error')
ax.set_title('Error convergence (piecewise linear FEM)')
ax.legend()

plt.tight_layout()
plt.show()

# Print convergence rates
print('Error convergence rates:')
print(f'{"N":>6}  {"h":>10}  {"L_inf error":>12}  {"L2 error":>12}  {"L_inf rate":>12}  {"L2 rate":>10}')
for i, N in enumerate(N_values):
    rate_inf = ''
    rate_l2 = ''
    if i > 0:
        rate_inf = f'{np.log(errors_Linf[i-1]/errors_Linf[i]) / np.log(h_values[i-1]/h_values[i]):.2f}'
        rate_l2 = f'{np.log(errors_L2[i-1]/errors_L2[i]) / np.log(h_values[i-1]/h_values[i]):.2f}'
    print(f'{N:6d}  {h_values[i]:10.6f}  {errors_Linf[i]:12.2e}  {errors_L2[i]:12.2e}  {rate_inf:>12}  {rate_l2:>10}')

In [ ]:
# ── Example 2: f(x) = 1 (constant), exact u(x) = x(1-x)/2 ──────────────────

f_func2 = lambda x: np.ones_like(x)
u_exact_func2 = lambda x: x * (1 - x) / 2

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_fine, u_exact_func2(x_fine), 'k-', lw=2.5, label='Exact: $x(1-x)/2$')

for i, N in enumerate([3, 6, 15, 50]):
    x_fem, u_fem = fem_1d(f_func2, N)
    ax.plot(x_fem, u_fem, 'o--', color=PALETTE[i], markersize=4, label=f'FEM, N={N}')

ax.set_title(r'FEM solution of $-u\prime\prime = 1$, $u(0)=u(1)=0$')
ax.set_xlabel('x')
ax.set_ylabel('u(x)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Example 3: f(x) = 10*pi^2 * sin(pi*x) + 100*pi^2 * sin(10*pi*x) ───────
# A multi-scale RHS to test FEM on a problem requiring more resolution.
# Exact: u(x) = sin(pi*x) + sin(10*pi*x)

f_func3 = lambda x: (np.pi**2) * np.sin(np.pi * x) + (10 * np.pi)**2 * np.sin(10 * np.pi * x)
u_exact_func3 = lambda x: np.sin(np.pi * x) + np.sin(10 * np.pi * x)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)

ax = axes[0]
ax.plot(x_fine, u_exact_func3(x_fine), 'k-', lw=1.5, label='Exact')
for i, N in enumerate([10, 30, 100]):
    x_fem, u_fem = fem_1d(f_func3, N)
    ax.plot(x_fem, u_fem, 'o-', color=PALETTE[i], markersize=3, label=f'FEM, N={N}')
ax.set_title('Multi-scale problem: FEM solutions')
ax.set_xlabel('x')
ax.set_ylabel('u(x)')
ax.legend()

# Convergence
ax = axes[1]
N_vals3 = [10, 20, 40, 80, 160, 320]
errs3 = []
hs3 = []
for N in N_vals3:
    x_fem, u_fem = fem_1d(f_func3, N)
    u_ex = u_exact_func3(x_fem)
    errs3.append(np.max(np.abs(u_fem - u_ex)))
    hs3.append(1.0 / (N + 1))

hs3 = np.array(hs3)
ax.loglog(hs3, errs3, 'o-', color=PALETTE[0], lw=2, label=r'$\|e\|_{L^\infty}$')
ax.loglog(hs3, 8.0 * hs3**2, 'k--', lw=1, alpha=0.5, label=r'$O(h^2)$')
ax.set_xlabel('$h$')
ax.set_ylabel('Error')
ax.set_title('Multi-scale: error convergence')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Visualize the FEM basis functions (hat functions) ─────────────────────────

N_demo = 6
h_demo = 1.0 / (N_demo + 1)
x_demo = np.linspace(0, 1, 1000)
nodes = np.linspace(0, 1, N_demo + 2)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_WIDE)

# Panel 1: individual hat functions
ax = axes[0]
for i in range(1, N_demo + 1):  # interior nodes only
    phi = np.zeros_like(x_demo)
    # Left ramp
    mask_left = (x_demo >= nodes[i-1]) & (x_demo <= nodes[i])
    phi[mask_left] = (x_demo[mask_left] - nodes[i-1]) / h_demo
    # Right ramp
    mask_right = (x_demo >= nodes[i]) & (x_demo <= nodes[i+1])
    phi[mask_right] = (nodes[i+1] - x_demo[mask_right]) / h_demo
    
    ax.plot(x_demo, phi, color=PALETTE[i % len(PALETTE)], lw=1.5,
            label=rf'$\phi_{i}$' if i <= 4 else None)
    ax.fill_between(x_demo, phi, alpha=0.08, color=PALETTE[i % len(PALETTE)])

for node in nodes:
    ax.axvline(node, ls=':', color='grey', lw=0.5, alpha=0.5)

ax.set_title(f'Piecewise linear hat basis (N={N_demo} interior nodes)')
ax.set_xlabel('x')
ax.set_ylabel(r'$\phi_i(x)$')
ax.legend(ncol=2, fontsize=9)

# Panel 2: stiffness matrix sparsity pattern
ax = axes[1]
K_dense = np.zeros((N_demo, N_demo))
for i in range(N_demo):
    K_dense[i, i] = 2.0 / h_demo
    if i > 0:
        K_dense[i, i-1] = -1.0 / h_demo
    if i < N_demo - 1:
        K_dense[i, i+1] = -1.0 / h_demo

im = ax.imshow(K_dense, cmap='RdBu_r', aspect='equal')
ax.set_title('Stiffness matrix $K$ (tridiagonal)')
ax.set_xlabel('Column $j$')
ax.set_ylabel('Row $i$')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

print('Stiffness matrix K (scaled by h):')
print(np.round(K_dense * h_demo, 2))

In [ ]:
# ── Verify Poincare inequality numerically ───────────────────────────────────
# For u in H^1_0(0,1):  ||u||_{L^2} <= C_P * ||u'||_{L^2}
# The best constant is C_P = 1/pi for the interval (0,1).

np.random.seed(123)
x = np.linspace(0, 1, 5000)
dx = x[1] - x[0]

ratios = []
labels_poincare = []

# Generate random functions in H^1_0(0,1) via random Fourier sine series
n_trials = 500
for trial in range(n_trials):
    # Random coefficients for sin(k*pi*x)
    K_max = np.random.randint(1, 30)
    coeffs = np.random.randn(K_max)
    
    u = np.zeros_like(x)
    du = np.zeros_like(x)
    for k in range(1, K_max + 1):
        u += coeffs[k-1] * np.sin(k * np.pi * x)
        du += coeffs[k-1] * k * np.pi * np.cos(k * np.pi * x)
    
    norm_u = np.sqrt(np.trapz(u**2, dx=dx))
    norm_du = np.sqrt(np.trapz(du**2, dx=dx))
    
    if norm_du > 1e-10:
        ratios.append(norm_u / norm_du)

C_P_theoretical = 1.0 / np.pi

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ratios, bins=50, color=PALETTE[0], alpha=0.7, edgecolor='white', density=True)
ax.axvline(C_P_theoretical, color=PALETTE[3], ls='--', lw=2.5,
           label=rf'$C_P = 1/\pi \approx {C_P_theoretical:.4f}$')
ax.axvline(max(ratios), color=PALETTE[1], ls=':', lw=2,
           label=f'Max observed: {max(ratios):.4f}')
ax.set_xlabel(r'$\|u\|_{L^2}\, /\, \|u\'\|_{L^2}$')
ax.set_ylabel('Density')
ax.set_title(r'Poincar\'e inequality: $\|u\|_{L^2} \leq C_P \|u\'\|_{L^2}$ for $u \in H^1_0(0,1)$')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Theoretical Poincare constant: C_P = 1/pi = {C_P_theoretical:.6f}')
print(f'Maximum observed ratio:        {max(ratios):.6f}')
print(f'All ratios <= C_P: {all(r <= C_P_theoretical + 1e-6 for r in ratios)}')

---
## Summary

| Concept | Key Idea |
|---------|----------|
| **Distributions** | Continuous linear functionals on test functions; extend the notion of "function" |
| **Distributional derivative** | $\langle T', \varphi \rangle = -\langle T, \varphi' \rangle$; every distribution is infinitely differentiable |
| **Weak derivative** | Distributional derivative that happens to be a regular distribution ($L^1_{\text{loc}}$ function) |
| **Sobolev space $W^{k,p}$** | $L^p$ functions whose weak derivatives up to order $k$ are also in $L^p$ |
| **$H^k = W^{k,2}$** | Hilbert space case; inner product involves all derivatives up to order $k$ |
| **Sobolev embedding** | $W^{k,p} \hookrightarrow C^m$ when $k - n/p > m$; regularity begets smoothness |
| **Lax-Milgram** | Existence and uniqueness for variational problems with coercive bilinear forms |
| **FEM** | Galerkin approximation in finite-dimensional subspace; $O(h^2)$ convergence for piecewise linear elements |

---
## References

1. **L. C. Evans**, *Partial Differential Equations*, 2nd ed., AMS, 2010. Chapters 5 (Sobolev spaces) and 6 (second-order elliptic equations).

2. **R. A. Adams & J. J. F. Fournier**, *Sobolev Spaces*, 2nd ed., Academic Press, 2003. The standard reference for Sobolev space theory.

3. **L. Schwartz**, *Theorie des distributions*, Hermann, Paris, 1950--1951. The foundational work on distribution theory.

4. **H. Brezis**, *Functional Analysis, Sobolev Spaces and Partial Differential Equations*, Springer, 2011. Excellent modern treatment combining functional analysis with PDE applications.

5. **S. C. Brenner & L. R. Scott**, *The Mathematical Theory of Finite Element Methods*, 3rd ed., Springer, 2008. Rigorous treatment of FEM in the Sobolev space framework.

6. **F. G. Friedlander & M. Joshi**, *Introduction to the Theory of Distributions*, 2nd ed., Cambridge University Press, 1998. Accessible introduction to distributions.

7. **E. Kreyszig**, *Introductory Functional Analysis with Applications*, Wiley, 1989. Chapter on distributions and weak derivatives accessible to engineers.